# 🤖 Hybrid Punjabi Speech-to-Text (Whisper) & Refinement (Gemma 4) Pipeline (Colab Edition)
This notebook implements a two-stage Punjabi transcription and script transliteration pipeline, optimized for the **Google Colab** environment with Google Drive integration:
1. **ASR (Speech-To-Text)**: Uses **Whisper-large-v3-turbo** to transcribe audio files into raw Gurmukhi Punjabi text.
2. **Repetition Compression**: Uses a programmatic cleaning filter to collapse Whisper's repeating loops or stutters.
3. **Verbatim Text Refinement**: Uses **Gemma 4 12B-it** to edit grammar errors and stutters while preserving exact words.
4. **Script Transliteration**: Uses **Gemma 4 12B-it** to transliterate Gurmukhi text phonetically into Devanagari.
5. **Memory-Optimization**: Sequentially loads Whisper and Gemma 4, releasing Whisper from RAM/GPU cache BEFORE loading Gemma 4 to prevent Colab crashes.
6. **Anti-Hallucination**: Employs separate `system` message blocks and **greedy decoding (`do_sample=False`)** to eliminate commentary and filler words.

---
### 🛠️ Step 1: Install Dependencies
Install the required packages (`transformers`, `accelerate`, `bitsandbytes`, `huggingface_hub`) on the Colab runtime environment.

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub

### 🔑 Step 2: Hugging Face Authentication
Authenticate with Hugging Face to obtain access to download the gated model weights.

In [ ]:
from huggingface_hub import login

# Paste your Hugging Face token here when prompted, or configure via Secrets
login()

### 📁 Step 3: Mount Google Drive
Mount Google Drive to access audio folders and write transcription output files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 📁 Step 4: Scan Folder and Select Audio File
Search the Google Drive directory for supported audio formats and display a dropdown widget.

In [ ]:
import os
import glob
import ipywidgets as widgets
from IPython.display import display

# Change 'Punjabi_Audio_Project' to your exact Google Drive folder name
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files"

# Scan for matching audio formats
audio_formats = ["*.wav", "*.mp3", "*.m4a", "*.flac"]
audio_file_paths = []
for fmt in audio_formats:
    audio_file_paths.extend(glob.glob(os.path.join(DRIVE_FOLDER_PATH, fmt)))

# Create a dictionary mapping the base file name to its full path
audio_files_dict = {os.path.basename(path): path for path in audio_file_paths}

if not audio_files_dict:
    print("No audio files found! Please check your Google Drive folder path or folder name.")
else:
    # Initialize and display the interactive dropdown widget
    audio_dropdown = widgets.Dropdown(
        options=list(audio_files_dict.keys()),
        description='Select Audio:',
        disabled=False,
        layout={'width': 'max-content'}
    )
    print(f"Successfully connected! Found {len(audio_file_paths)} audio files.")
    display(audio_dropdown)

### 🗣️ Step 5: Run Whisper ASR (Stage 1)
Transcribes the audio into Punjabi Gurmukhi text. To prevent GPU memory crashes, **we completely unload Whisper from memory and flush the GPU cache** immediately after transcription finishes, preparing the environment for Gemma 4.

In [ ]:
import torch
import librosa
from transformers import pipeline
import gc

selected_file = audio_dropdown.value
if not selected_file:
    print("Please select a file from the dropdown above.")
else:
    file_path = audio_files_dict[selected_file]
    print(f"--- Processing: {selected_file} ---")

    # 1. Transcribe Audio using Whisper (Timestamps disabled to prevent pipeline crash)
    print("⏳ Step 1: Transcribing audio via Whisper...")
    try:
        whisper_pipe = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device="cuda" if torch.cuda.is_available() else "cpu",
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
        )

        print("⏳ Decoding and resampling audio...")
        audio_array, sr = librosa.load(file_path, sr=16000)

        print("⏳ Processing transcription payload... ")
        result = whisper_pipe(
            {"array": audio_array, "sampling_rate": sr},
            return_timestamps=False,             # 🛠️ FIXED: Disabling this stops the NoneType comparison crash
            chunk_length_s=30,
            generate_kwargs={
                "language": "punjabi",
                "condition_on_prev_tokens": False
            }
        )
        raw_extracted_text = result["text"]
        print("✅ Raw transcription complete.")
    except Exception as e:
        print(f"❌ Audio Transcription Error: {e}")
        raw_extracted_text = None

    # 2. Cleanup Whisper pipeline memory from GPU and system RAM
    if 'whisper_pipe' in locals():
        del whisper_pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Successfully deleted Whisper pipeline and flushed GPU VRAM cache to prepare for Gemma 4.")

### 🧠 Step 6: Load Gemma 4 12B-it Model
Initialize the tokenizer and load the 4-bit quantized Gemma 4 model using HF Secrets (`HF_TOKEN`) automatically. We load it with **`low_cpu_mem_usage=True`** to sequentially load layers and bypass Colab's system RAM limits.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from huggingface_hub import login

# Updated to Gemma 4 12B
model_id = "google/gemma-4-12b-it"

try:
    # 1. Attempt to get token from Secrets
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("✅ Logged in to Hugging Face via Secrets.")
    else:
        print("❌ HF_TOKEN not found in Secrets. Please add it to the Key icon menu.")
except Exception as e:
    print(f"❌ Error accessing Secrets: {e}")
    hf_token = None

if hf_token:
    # 2. Setup Quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    print(f"⏳ Downloading {model_id} (this takes a moment)... ")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            low_cpu_mem_usage=True,  # 🛠️ FIXED: Sequential layer loading avoids system RAM OOM crash
            token=hf_token
        )
        print("✅ Model and Tokenizer loaded successfully!")
    except Exception as e:
        print(f"❌ Loading failed: {e}")
        print(f"Make sure you accepted the license at https://huggingface.co/{model_id}")

### ✍️ Step 7: Refine Transcription Text (Stage 2)
Programmatically compresses repetition loops and then refines the transcript with Gemma 4 using greedy decoding (`do_sample=False`) to avoid hallucinations.

In [ ]:
if 'raw_extracted_text' not in locals() or not raw_extracted_text:
    print("Error: No raw Whisper transcription found. Please run the Whisper transcription cell first.")
elif 'tokenizer' not in locals() or 'model' not in locals():
    print("❌ Error: Gemma model not loaded. Run the initialization cell first.")
else:
    # 1. Native Loop Compression Layer
    print("⏳ Step 2: Compressing infinite text loops...")

    def compress_loops(text):
        words = text.split()
        if not words: return ""
        cleaned = [words[0]]
        consecutive_count = 1
        for word in words[1:]:
            if word == cleaned[-1]:
                consecutive_count += 1
                if consecutive_count <= 2:  # Keeps natural Punjabi double words like 'ਹੌਲੀ ਹੌਲੀ'
                    cleaned.append(word)
            else:
                consecutive_count = 1
                cleaned.append(word)
        return " ".join(cleaned)

    cleaned_raw_text = compress_loops(raw_extracted_text)

    # 2. Refine with Gemma
    print("⏳ Step 3: Refining text with Gemma...")
    system_instruction = (
        "You are an expert Punjabi editor. Your task is to take the provided transcription, "
        "fix grammar errors, and remove any remaining minor stutters while keeping the actual "
        "spoken content verbatim. Output ONLY the cleaned Punjabi text. Do not include commentary."
    )

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"Raw Text to Clean:\n{cleaned_raw_text}"}
    ]

    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=False  # Crucial: Use greedy decoding to prevent hallucinations
            )

        final_transcript = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        print(f"\n✨ Final Cleaned Transcript:\n{final_transcript}")
    except Exception as e:
        print(f"❌ Gemma Refinement Error: {e}")

### 💾 Step 8: Save Gurmukhi Transcription to Google Drive
Saves the finalized verbatim transcript file to Google Drive folder.

In [ ]:
import os

# 1. Define the base directory for saving outputs
base_save_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Using Gemma Models"

# 2. Check if a file was actually processed in the previous step
if 'selected_file' not in locals() or 'final_transcript' not in locals():
    print("Error: No processed transcript found. Please run the transcription cell first.")
else:
    # 3. Create a unique ID from the filename (removing extension)
    file_id = os.path.splitext(selected_file)[0]

    # 4. Dynamically create the target folder
    target_folder = os.path.join(base_save_path, file_id)
    os.makedirs(target_folder, exist_ok=True)

    # 5. Define the filename: lowercase ID + _gurumukhi.txt
    file_save_name = f"{file_id.lower()}_gurumukhi.txt"
    transcript_path = os.path.join(target_folder, file_save_name)

    # 6. Save the content
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(final_transcript)

    print(f"✨ Success! Verbatim transcript saved to: {transcript_path}")

### 🕉️ Step 9: Transliterate Gurmukhi script to Devanagari script
Converts the Gurmukhi script into Hindi Devanagari script using Gemma 4, applying strict dialect guidelines and greedy decoding.

In [ ]:
import os

# 1. Guard rails to ensure previous cell data is present
if 'selected_file' not in locals() or 'final_transcript' not in locals():
    print("Error: No processed transcript found. Please run the transcription cell first.")
else:
    print("Converting Gurmukhi transcript to Devanagari script...")

    # 2. Prepare the transliteration prompt for Gemma
    system_instruction = (
        "You are a high-precision linguistic conversion model. Transliterate the following text from Gurmukhi script (Punjabi) into Devanagari script (Hindi characters).\n\n"
        "Strict Rules:\n"
        "1. Do NOT translate the language into formal textbook Hindi. Keep the exact spoken words, colloquial expressions, and dialect intact—only change the script alphabets (e.g., 'ਵੀਰ ਜੀ, ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਜੀ' must become 'वीर जी, सत श्री अकाल जी।').\n"
        "2. Preserve all structural formatting, speaker tags, and markdown characters exactly as they appear.\n"
        "3. Output ONLY the transliterated Devanagari text. Do not include any explanations, preambles, or commentary."
    )

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"Gurmukhi Source Text:\n{final_transcript}"}
    ]

    # 3. Apply template and generate using the local model
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048, # Ensure enough capacity for complete transliteration
            do_sample=False      # Crucial: Use greedy decoding to prevent hallucinations
        )

    devanagari_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    # 4. Save to Drive
    file_id = os.path.splitext(selected_file)[0]
    target_folder = os.path.join(base_save_path, file_id)
    os.makedirs(target_folder, exist_ok=True)

    file_save_name = f"{file_id.lower()}_devanagari.txt"
    transcript_path = os.path.join(target_folder, file_save_name)

    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(devanagari_text)

    print(f"✨ Success! Script converted and file saved to: {transcript_path}")

### 📝 Step 10: Verify Model Functionality
Runs a simple test prompt to ensure that the loaded model responds correctly.

In [ ]:
# Create a test prompt
messages = [
    {"role": "user", "content": "Hello! Can you help me transcribe and analyze Punjabi text from audio snippets?"}
]

# Apply Gemma's standard chat template
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False  # Greedy decoding
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print(response)